In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

master_df = spark.read.csv(
    "/Volumes/workspace/default/myvolume/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

display(master_df)

from pyspark.sql.functions import col, sum, when

"""master_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in master_df.columns
]).show()"""
clean_df = master_df.dropna()
clean_df = clean_df.dropDuplicates()
print("Rows After Cleaning :", clean_df.count())

# Replace spaces with underscores
clean_df = clean_df.toDF(*[
    col.replace(" ", "_")
    for col in clean_df.columns
])

display(clean_df)

clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("superstore_master")
display(spark.table("superstore_master"))

deltaTable = DeltaTable.forName(
    spark,
    "superstore_master"
)

incremental_df = spark.sql("""
SELECT *
FROM superstore_master
LIMIT 5
""")

display(incremental_df)

from pyspark.sql.functions import when, lit

incremental_df = incremental_df.withColumn(
    "Customer_Name",
    when(
        col("Order_ID") == "US-2015-108966",
        lit("Sean O'Donnell (Updated)")
    ).otherwise(col("Customer_Name"))
)

display(incremental_df)


new_record = spark.sql("""
SELECT *
FROM superstore_master
LIMIT 1
""")

new_record = new_record \
    .withColumn("Order_ID", lit("CA-2026-999999")) \
    .withColumn("Customer_Name", lit("John Smith"))

display(new_record)

incremental_df = incremental_df.union(new_record)

display(incremental_df)

deltaTable.alias("target").merge(
    incremental_df.alias("source"),
    "target.Order_ID = source.Order_ID"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

final_df = spark.table("superstore_master")

display(final_df)




Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
5,CA-2026-999999,2015-10-11,2015-10-18,Standard Class,SO-20335,John Smith,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell (Updated),Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"""Stur-D-Stor Shelving, Vertical 5-Shelf: 72""""H x 36""""W x 18 1/2""""D""",665.88,6,0,13.3176
21,CA-2014-143336,2014-08-27,2014-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109,West,OFF-BI-10002215,Office Supplies,Binders,"Wilson Jones Hanging View Binder, White, 1""""",22.72,4,0.2,7.384
53,CA-2015-115742,2015-04-18,2015-04-22,Standard Class,DP-13000,Darren Powers,Consumer,United States,New Albany,Indiana,47150,Central,FUR-CH-10003061,Furniture,Chairs,"Global Leather Task Chair, Black",89.99,1,0,17.0981
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0,165.3813
5,CA-2026-999999,2015-10-11,2015-10-18,Standard Class,SO-20335,John Smith,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
